## The ESG company-year view

The ESG source data is delivered in wide format, with one row per company and separate columns for multiple ESG indicators across calendar years.

To align ESG information with the contract-year dataset and to support later weak supervision and meta-learning steps, the ESG table is reshaped into a company-year structure:

**one row = one company in one year**

Only absolute calendar-year columns are used for this transformation (for example 2022–2026), because they provide a stable temporal key for joining with other yearly datasets such as contracts and financials.

A `Join_Year` column is created as:

`Join_Year = year + 1`

This follows the same leakage-prevention logic used in our pipeline, where company information from year *t* is used to explain contract outcomes in year *t + 1*.

In [40]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

project_root: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-
src_path: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/src


In [41]:
import pandas as pd
import numpy as np
import re


In [42]:
df_esg = pd.read_csv(
    "/Users/annita/Desktop/Thesis/tables/moodys/ESG_Data_AXEF.csv",
    header=0,
    skiprows=[1]
)

In [43]:
print(df_esg.shape)
print(df_esg.columns.tolist())
df_esg.head()

(1893, 104)
['Company name Latin alphabet', 'Risk level', 'BvD ID number', 'BvD sectors', 'MSCI - Overall indicative ESG score\nLast avail. yr', 'MSCI - Overall indicative ESG score\nYear - 1', 'MSCI - Overall indicative ESG score\nYear - 2', 'MSCI - Overall indicative ESG score\nYear - 3', 'MSCI - Overall indicative ESG score\nYear - 4', 'MSCI - Overall indicative ESG score\n2026', 'MSCI - Overall indicative ESG score\n2025', 'MSCI - Overall indicative ESG score\n2024', 'MSCI - Overall indicative ESG score\n2023', 'MSCI - Overall indicative ESG score\n2022', 'MSCI - Final indicative ESG industry-adjusted score\nLast avail. yr', 'MSCI - Final indicative ESG industry-adjusted score\nYear - 1', 'MSCI - Final indicative ESG industry-adjusted score\nYear - 2', 'MSCI - Final indicative ESG industry-adjusted score\nYear - 3', 'MSCI - Final indicative ESG industry-adjusted score\nYear - 4', 'MSCI - Final indicative ESG industry-adjusted score\n2026', 'MSCI - Final indicative ESG industry-adju

,Company name Latin alphabet,Risk level,BvD ID number,BvD sectors,MSCI - Overall indicative ESG score\nLast avail. yr,MSCI - Overall indicative ESG score\nYear - 1,MSCI - Overall indicative ESG score\nYear - 2,MSCI - Overall indicative ESG score\nYear - 3,MSCI - Overall indicative ESG score\nYear - 4,MSCI - Overall indicative ESG score\n2026,...,MSCI - Indicative ESG score industry maximum\nLast avail. yr,MSCI - Indicative ESG score industry maximum\nYear - 1,MSCI - Indicative ESG score industry maximum\nYear - 2,MSCI - Indicative ESG score industry maximum\nYear - 3,MSCI - Indicative ESG score industry maximum\nYear - 4,MSCI - Indicative ESG score industry maximum\n2026,MSCI - Indicative ESG score industry maximum\n2025,MSCI - Indicative ESG score industry maximum\n2024,MSCI - Indicative ESG score industry maximum\n2023,MSCI - Indicative ESG score industry maximum\n2022
0,Jeev Lifeworks LLP,Do not source,IN0038742908,Business Services,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,MY422033-W,"Chemicals, Petroleum, Rubber & Plastic",5,5,5,n.a.,n.a.,n.a.,...,7,7,7,n.a.,n.a.,n.a.,7,7,7,n.a.
2,Daicel Corporation,Go ahead,JP4120001125937,"Chemicals, Petroleum, Rubber & Plastic",4,4,n.a.,n.a.,n.a.,n.a.,...,6,6,n.a.,n.a.,n.a.,n.a.,6,6,n.a.,n.a.
3,"Polyplastics Co.,Ltd.",Take caution,JP1010401053834,"Chemicals, Petroleum, Rubber & Plastic",3,3,n.a.,n.a.,n.a.,n.a.,...,6,6,n.a.,n.a.,n.a.,n.a.,6,6,n.a.,n.a.
4,"Kolon Industries, Inc.",Go ahead,KR1353110013606,"Chemicals, Petroleum, Rubber & Plastic",3,4,n.a.,n.a.,n.a.,n.a.,...,6,6,n.a.,n.a.,n.a.,n.a.,n.a.,6,6,n.a.


In [44]:
df_esg_work = df_esg.copy()

# Rename identifier columns
df_esg_work = df_esg_work.rename(columns={
    "Company name Latin alphabet": "company_name",
    "Risk level": "risk_level",
    "BvD ID number": "moodys_bvd_id",
    "BvD sectors": "sector"
})

# Clean key
df_esg_work["moodys_bvd_id"] = (
    df_esg_work["moodys_bvd_id"]
    .astype("string")
    .str.strip()
)

# Replace Moody's missing strings
df_esg_work = df_esg_work.replace(["n.a.", "n.a", "N/A", ""], pd.NA)

**Select absolute year columns**

The ESG dataset contains both relative-year fields (e.g., "Last avail. yr", "Year - 1") and absolute calendar-year fields (e.g., 2022–2026).

For integration with the contract-year dataset, only `absolute` calendar-year columns are retained.  
These provide a stable temporal reference and allow consistent alignment across datasets.

In [45]:
absolute_year_pattern = re.compile(r"(.*)\n(20\d{2})$")

melt_cols = [
    c for c in df_esg_work.columns
    if absolute_year_pattern.search(c)
]

print("Absolute-year ESG columns detected:", len(melt_cols))

Absolute-year ESG columns detected: 50


**Reshape ESG data to long format**

The dataset is transformed from wide to `long format` using a melt operation.

This step converts multiple year-specific columns into a standardized structure where each observation represents:

one company  
one ESG metric  
one calendar year  

This structure simplifies aggregation and supports later integration into a company-year analytical view.

Note: Wide table  → Melt → Long table & Many year columns → One year column

In [46]:
# Melt to long format
# one row = one company, one metric, one year
df_esg_long = pd.melt(
    df_esg_work,
    id_vars=["company_name", "moodys_bvd_id", "risk_level", "sector"],
    value_vars=melt_cols,
    var_name="raw_metric",
    value_name="value"
)

# Extract metric name and year
df_esg_long[["metric_name", "year"]] = df_esg_long["raw_metric"].str.extract(r"(.*)\n(20\d{2})$")

df_esg_long["year"] = pd.to_numeric(df_esg_long["year"], errors="coerce").astype("Int64")

**Clean metric names and convert values**

ESG metric names are standardized to concise variable names to improve readability and consistency across datasets.

All ESG values are converted to numeric format, and non-numeric placeholders (e.g., "n.a.") are treated as missing values.

In [47]:
metric_map = {
    "MSCI - Overall indicative ESG score": "esg_overall",
    "MSCI - Final indicative ESG industry-adjusted score": "esg_industry_adjusted",
    "MSCI - Environmental pillar indicative score": "env_score",
    "MSCI - Environmental pillar indicative weight": "env_weight",
    "MSCI - Social pillar indicative score": "social_score",
    "MSCI - Social pillar indicative weight": "social_weight",
    "MSCI - Governance pillar indicative score": "gov_score",
    "MSCI - Governance pillar indicative weight": "gov_weight",
    "MSCI - Indicative ESG score industry minimum": "industry_min",
    "MSCI - Indicative ESG score industry maximum": "industry_max",
}

df_esg_long["metric_name"] = df_esg_long["metric_name"].replace(metric_map)

# Convert values to numeric where possible
df_esg_long["value"] = pd.to_numeric(df_esg_long["value"], errors="coerce")

**Create company-year ESG dataset**

The long-format dataset is pivoted to produce a company-year structure.

In this final structure:

one row represents one company in one year  

Each ESG indicator becomes a separate feature column.  
This structure aligns with the contract-year dataset and supports downstream modeling and feature generation.

In [48]:
# Pivot back to company-year
#   one row = one company in one year
df_esg_yearly = (
    df_esg_long
    .pivot_table(
        index=["moodys_bvd_id", "company_name", "risk_level", "sector", "year"],
        columns="metric_name",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

df_esg_yearly.columns.name = None

**Create Join_Year variable**

To prevent information leakage in predictive modeling, ESG information from year t is linked to contract outcomes in year t + 1.

A Join_Year variable is therefore created as:

Join_Year = year + 1

example:

ESG 2022 → explains contract risk in 2023

ESG 2023 → explains contract risk in 2024

In [49]:
df_esg_yearly["Join_Year"] = df_esg_yearly["year"] + 1

# Reorder columns
front_cols = [
    "moodys_bvd_id",
    "company_name",
    "risk_level",
    "sector",
    "year",
    "Join_Year",
    "esg_overall",
    "esg_industry_adjusted",
    "env_score",
    "env_weight",
    "social_score",
    "social_weight",
    "gov_score",
    "gov_weight",
    "industry_min",
    "industry_max",
]

existing_front_cols = [c for c in front_cols if c in df_esg_yearly.columns]
remaining_cols = [c for c in df_esg_yearly.columns if c not in existing_front_cols]

df_esg_yearly = df_esg_yearly[existing_front_cols + remaining_cols]


# Checks
print("Original ESG shape:", df_esg.shape)
print("Long ESG shape:", df_esg_long.shape)
print("Final ESG yearly shape:", df_esg_yearly.shape)

print("\nDuplicate company-year rows:",
      df_esg_yearly.duplicated(["moodys_bvd_id", "year"]).sum())

print("\nUnique companies:",
      df_esg_yearly["moodys_bvd_id"].nunique())

print("\nYear range:")
print(df_esg_yearly["year"].min(), "to", df_esg_yearly["year"].max())

df_esg_yearly.head()

Original ESG shape: (1893, 104)
Long ESG shape: (94650, 8)
Final ESG yearly shape: (2093, 16)

Duplicate company-year rows: 0

Unique companies: 1027

Year range:
2022 to 2025


,moodys_bvd_id,company_name,risk_level,sector,year,Join_Year,esg_overall,esg_industry_adjusted,env_score,env_weight,social_score,social_weight,gov_score,gov_weight,industry_min,industry_max
0,AE0043368774,Fifth Element Event Management L.L.C,Take caution,Business Services,2023,2024,5.0,5.0,9.0,5.0,6.0,28.0,5.0,67.0,3.0,8.0
1,AT9070096767,Zauner Anlagentechnik GmbH,Go ahead,"Industrial, Electric & Electronic Machinery",2023,2024,3.0,1.0,3.0,33.0,3.0,20.0,5.0,47.0,3.0,7.0
2,AT9070096767,Zauner Anlagentechnik GmbH,Go ahead,"Industrial, Electric & Electronic Machinery",2024,2025,3.0,1.0,3.0,33.0,1.0,20.0,5.0,47.0,3.0,7.0
3,AT9070096767,Zauner Anlagentechnik GmbH,Go ahead,"Industrial, Electric & Electronic Machinery",2025,2026,3.0,1.0,3.0,33.0,1.0,20.0,5.0,47.0,3.0,7.0
4,AT9110192427,Flextronics International Gesellschaft m.b.H.,Do not source,Metals & Metal Products,2023,2024,4.0,1.0,4.0,40.0,4.0,25.0,5.0,35.0,4.0,7.0


Comment: After melt we have 94650 col because: 1893 companies × ~50 ESG columns and after pivot 2093 company-year rows, about 2 years per company on average

In [50]:
df_esg_yearly["Join_Year"].value_counts().sort_index()

Join_Year
2023    186
2024    897
2025    819
2026    191
Name: count, dtype: Int64

In [51]:
from master_thesis.config import DATA_RAW

df_esg_yearly.to_csv(DATA_RAW / "esg_yearly.csv", index=False)
print("Saved to:", DATA_RAW / "esg_yearly.csv")


Saved to: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/raw/esg_yearly.csv
